# A/B Testing Practice Lab: Interpretation and Decision-Making

This notebook is the practice companion to `01_AB_testing.ipynb` and `02_AB_testing_experiment_design.ipynb`.
It is designed to help you rehearse the workflow from those notebooks with a stronger focus on **interpretation**, **decision-making**, and **communicating results clearly**.

How to use this notebook:

- Try each exercise cell first.
- Keep the `TODO` markers until you are ready to solve them.
- Check the `### Solution` section right after each exercise.
- Compare not only the code, but also the interpretation.


In [ ]:
import math

import pandas as pd
from IPython.display import Markdown, display
from scipy.stats import chi2_contingency
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize


def clean_experiment_data(raw_df: pd.DataFrame) -> pd.DataFrame:
    """Apply the same cleaning logic used in the first two notebooks."""
    users_both_groups = raw_df.groupby("user_id")["group"].nunique()
    users_both_groups = users_both_groups[users_both_groups > 1].index

    users_both_pages = raw_df.groupby("user_id")["landing_page"].nunique()
    users_both_pages = users_both_pages[users_both_pages > 1].index

    contaminated_users = set(users_both_groups) | set(users_both_pages)

    cleaned_df = raw_df[~raw_df["user_id"].isin(contaminated_users)].copy()
    first_exposure = cleaned_df.groupby("user_id")["timestamp"].min().reset_index()
    cleaned_df = cleaned_df.merge(first_exposure, on=["user_id", "timestamp"])
    cleaned_df["timestamp"] = pd.to_datetime(cleaned_df["timestamp"])
    return cleaned_df


def required_sample_size_per_variant(
    baseline_rate: float,
    relative_lift: float,
    alpha: float = 0.05,
    power: float = 0.80,
) -> int:
    """Return the required users per variant for a two-sample conversion test."""
    power_model = NormalIndPower()
    target_rate = baseline_rate * (1 + relative_lift)
    effect_size = proportion_effectsize(target_rate, baseline_rate)
    required_n = power_model.solve_power(
        effect_size=effect_size,
        alpha=alpha,
        power=power,
        ratio=1.0,
        alternative="two-sided",
    )
    return math.ceil(required_n)


def runtime_days(
    sample_size_per_variant: int,
    avg_daily_total: float,
    traffic_split: float = 0.5,
) -> float:
    """Convert a sample-size target into expected test duration."""
    return sample_size_per_variant / (avg_daily_total * traffic_split)


raw_df = pd.read_csv("data/ab_data.csv")
official_clean_df = clean_experiment_data(raw_df)
daily_traffic = official_clean_df.groupby(official_clean_df["timestamp"].dt.date).size()
control_df = official_clean_df[official_clean_df["group"] == "control"].copy()
treatment_df = official_clean_df[official_clean_df["group"] == "treatment"].copy()
baseline_rate = control_df["converted"].mean()
avg_daily_total = daily_traffic.mean()

print(f"Raw rows loaded: {raw_df.shape[0]:,}")
print(f"Clean rows available for solutions: {official_clean_df.shape[0]:,}")

## Exercise 1: Read the Dataset Like an Analyst

**Learning goal:** Understand the raw data before doing any statistical work.

**Task:**
1. Identify the number of rows in the raw dataset.
2. Fill in the number of unique users.
3. Write a short description for each column.

**Expected output:** A few summary values plus a small table or dictionary describing the columns.


In [ ]:
# TODO 1: Replace None with the number of unique users.
unique_users = None

# TODO 2: Replace each placeholder with a short column description.
column_descriptions = {
    "user_id": "TODO",
    "timestamp": "TODO",
    "group": "TODO",
    "landing_page": "TODO",
    "converted": "TODO",
}

print(f"Raw rows: {raw_df.shape[0]:,}")
print(f"Unique users: {unique_users}")
display(
    pd.DataFrame(
        column_descriptions.items(), columns=pd.Index(["column", "description"])
    )
)
raw_df.head()

### Solution


In [ ]:
solution_column_descriptions = {
    "user_id": "Unique identifier for a user in the experiment",
    "timestamp": "Time of the user's recorded exposure",
    "group": "Assigned experiment group: control or treatment",
    "landing_page": "Page version the user saw: old_page or new_page",
    "converted": "Binary outcome showing whether the user converted",
}

print(f"Raw rows: {raw_df.shape[0]:,}")
print(f"Unique users: {raw_df['user_id'].nunique():,}")
display(
    pd.DataFrame(
        solution_column_descriptions.items(),
        columns=pd.Index(["column", "description"]),
    )
)
raw_df.head()

## Exercise 2: Rebuild the Cleaning Logic

**Learning goal:** Recreate the core data-cleaning step used in the earlier notebooks.

**Task:**
1. Find users who appear in more than one group.
2. Find users who see more than one landing page.
3. Remove those contaminated users.
4. Keep only each user's first exposure.

**Expected output:** A cleaned dataframe and a short summary of cleaned rows and unique users.


In [ ]:
# TODO: Replace the placeholders with the real cleaning logic.
users_both_groups = pd.Index([])
users_both_pages = pd.Index([])
contaminated_users = set()

cleaned_practice_df = raw_df.copy()

print(f"Users in more than one group: {len(users_both_groups):,}")
print(f"Users who saw more than one page: {len(users_both_pages):,}")
print(f"Rows after cleaning: {cleaned_practice_df.shape[0]:,}")
print(f"Unique users after cleaning: {cleaned_practice_df['user_id'].nunique():,}")

### Solution


In [ ]:
users_both_groups = raw_df.groupby("user_id")["group"].nunique()
users_both_groups = users_both_groups[users_both_groups > 1].index

users_both_pages = raw_df.groupby("user_id")["landing_page"].nunique()
users_both_pages = users_both_pages[users_both_pages > 1].index

contaminated_users = set(users_both_groups) | set(users_both_pages)

cleaned_practice_df = raw_df[~raw_df["user_id"].isin(contaminated_users)].copy()
first_exposure = cleaned_practice_df.groupby("user_id")["timestamp"].min().reset_index()
cleaned_practice_df = cleaned_practice_df.merge(
    first_exposure, on=["user_id", "timestamp"]
)
cleaned_practice_df["timestamp"] = pd.to_datetime(cleaned_practice_df["timestamp"])

print(f"Users in more than one group: {len(users_both_groups):,}")
print(f"Users who saw more than one page: {len(users_both_pages):,}")
print(f"Rows after cleaning: {cleaned_practice_df.shape[0]:,}")
print(f"Unique users after cleaning: {cleaned_practice_df['user_id'].nunique():,}")
print(
    f"Matches the established cleaned row count: {cleaned_practice_df.shape[0] == official_clean_df.shape[0]}"
)

## Exercise 3: Compare Control and Treatment Rates

**Learning goal:** Move from cleaned data to a practical business comparison.

**Task:**
1. Compute the conversion rate for the control group.
2. Compute the conversion rate for the treatment group.
3. Compute the relative difference between treatment and control.
4. Write one sentence saying whether the observed gap looks practically meaningful.

**Expected output:** A few numeric values and a short interpretation.


In [ ]:
control_subset = official_clean_df[official_clean_df["group"] == "control"]
treatment_subset = official_clean_df[official_clean_df["group"] == "treatment"]

# TODO: Replace the placeholders below.
control_rate = None
treatment_rate = None
relative_gap = None
interpretation = "TODO: Explain whether this looks practically meaningful."

print(f"Control conversion rate: {control_rate}")
print(f"Treatment conversion rate: {treatment_rate}")
print(f"Relative gap: {relative_gap}")
print(interpretation)

### Solution


In [ ]:
control_rate = control_subset["converted"].mean()
treatment_rate = treatment_subset["converted"].mean()
relative_gap = (treatment_rate - control_rate) / control_rate

print(f"Control conversion rate: {control_rate:.2%}")
print(f"Treatment conversion rate: {treatment_rate:.2%}")
print(f"Relative gap: {relative_gap:.2%}")

interpretation = (
    "The treatment page performs slightly worse than control, and the gap is small "
    "in absolute terms, so it does not look like a strong practical improvement."
)
print(interpretation)

## Exercise 4: Reconstruct a Frequentist Test

**Learning goal:** Connect conversion rates to a formal hypothesis test.

**Task:**
1. Build a 2x2 contingency table for control vs treatment and converted vs not converted.
2. Run a chi-square test.
3. Explain what the p-value means.
4. Explain one thing the p-value does **not** mean.

**Expected output:** A contingency table, the chi-square statistic, the p-value, and a short interpretation.


In [ ]:
# TODO: Fill in the contingency table and the test result.
contingency_table = None
chi2_stat = None
p_value = None

print("Contingency table:")
print(contingency_table)
print(f"Chi-square statistic: {chi2_stat}")
print(f"p-value: {p_value}")
print("TODO: Add a sentence describing what the p-value means.")
print("TODO: Add a sentence describing what the p-value does not mean.")

### Solution


In [ ]:
control_converted = int(control_df["converted"].sum())
control_not_converted = int(control_df["converted"].count() - control_converted)
treatment_converted = int(treatment_df["converted"].sum())
treatment_not_converted = int(treatment_df["converted"].count() - treatment_converted)

contingency_table = pd.DataFrame(
    {
        "converted": [control_converted, treatment_converted],
        "not_converted": [control_not_converted, treatment_not_converted],
    },
    index=pd.Index(["control", "treatment"]),
)

chi2_stat, p_value, _, _ = chi2_contingency(contingency_table, correction=False)

display(contingency_table)
print(f"Chi-square statistic: {chi2_stat:.4f}")
print(f"p-value: {p_value:.4f}")

display(
    Markdown(
        "The p-value measures how surprising a difference this large would be if "
        "there were truly no conversion difference between the groups. It does not "
        "measure the probability that the null hypothesis is true, and it does not "
        "prove that the two versions are identical."
    )
)

## Exercise 5: Plan a Future Experiment

**Learning goal:** Turn a business target into a sample-size and runtime requirement.

**Task:**
1. Assume the business wants to detect a **2% relative lift**.
2. Use the helper functions to estimate the required users per variant.
3. Estimate the runtime using the observed average daily traffic.
4. Decide whether that target looks feasible with the traffic in this dataset.

**Expected output:** A target-lift summary and a short feasibility statement.


In [ ]:
target_relative_lift = 0.02

# TODO: Replace the placeholders below.
required_users = None
estimated_days = None
feasibility_statement = "TODO: Is this feasible with the observed traffic?"

print(f"Target relative lift: {target_relative_lift:.0%}")
print(f"Required users per variant: {required_users}")
print(f"Estimated runtime in days: {estimated_days}")
print(feasibility_statement)

### Solution


In [ ]:
target_relative_lift = 0.02
required_users = required_sample_size_per_variant(baseline_rate, target_relative_lift)
estimated_days = runtime_days(required_users, avg_daily_total)
observed_users_per_variant = min(control_df.shape[0], treatment_df.shape[0])

print(f"Target relative lift: {target_relative_lift:.0%}")
print(f"Required users per variant: {required_users:,}")
print(f"Estimated runtime in days: {estimated_days:.1f}")
print(f"Observed users per variant in this dataset: {observed_users_per_variant:,}")

if observed_users_per_variant >= required_users:
    feasibility_statement = "This looks feasible within the observed experiment size."
else:
    feasibility_statement = (
        "This target is not fully feasible within the observed experiment size; "
        "the dataset does not contain enough users per variant to detect a 2% lift "
        "with the chosen alpha and power."
    )

print(feasibility_statement)

## Exercise 6: Compare Two Business Scenarios

**Learning goal:** Translate statistical constraints into a product recommendation.

**Task:**
Compare these two cases:

- Scenario A: the business only cares about a **1% relative lift**
- Scenario B: the business cares about a **3% relative lift**

For each scenario, estimate users per variant and runtime. Then write a short recommendation about which scenario is more realistic for the current traffic level.

**Expected output:** A two-row comparison table plus a recommendation.


In [ ]:
scenario_lifts = [0.01, 0.03]
scenario_rows = []

# TODO: Fill the loop so each lift gets a sample-size and runtime estimate.
for lift in scenario_lifts:
    scenario_rows.append(
        {
            "relative_lift": lift,
            "required_users_per_variant": None,
            "estimated_runtime_days": None,
        }
    )

scenario_df = pd.DataFrame(scenario_rows)
display(scenario_df)
print("TODO: Write a recommendation comparing the 1% and 3% cases.")

### Solution


In [ ]:
scenario_lifts = [0.01, 0.03]
scenario_rows = []

for lift in scenario_lifts:
    required_users = required_sample_size_per_variant(baseline_rate, lift)
    estimated_days = runtime_days(required_users, avg_daily_total)
    scenario_rows.append(
        {
            "relative_lift": f"{lift:.0%}",
            "required_users_per_variant": required_users,
            "estimated_runtime_days": round(estimated_days, 1),
        }
    )

scenario_df = pd.DataFrame(scenario_rows)
display(scenario_df)

display(
    Markdown(
        "A **1%** lift target is much harder to detect and would require a very long "
        "experiment. A **3%** lift target is more realistic for the current traffic "
        "because the required sample size and runtime are much smaller. If the business "
        "needs fast decisions, the 3% scenario is the more practical planning target."
    )
)

## Exercise 7: Optional Challenge - Choose the Right Workflow

**Learning goal:** Decide when to use each notebook in a real project.

**Task:**
Imagine a product manager asks, *"Should we launch a new design experiment next month, and how would we evaluate it?"*

Write a short answer describing:

1. Which notebook you would start with first
2. Which notebook you would use after data collection
3. Why that order makes sense

**Expected output:** A short written recommendation, ideally 3-5 sentences.


In [ ]:
starting_notebook = "TODO"
follow_up_notebook = "TODO"
reasoning = "TODO: Explain the order in 3-5 sentences."

print(f"Start with: {starting_notebook}")
print(f"Then use: {follow_up_notebook}")
print(reasoning)

### Solution


In [ ]:
starting_notebook = "02_AB_testing_experiment_design.ipynb"
follow_up_notebook = "01_AB_testing.ipynb"
reasoning = (
    "Start with the experiment-design notebook because it helps define whether the "
    "test is worth running, how much traffic is needed, and how long the experiment "
    "should last. After data collection, use the analysis notebook to clean the data, "
    "compare control and treatment, and interpret the statistical results. This order "
    "keeps the team from launching an underpowered test and then misreading the outcome."
)

print(f"Start with: {starting_notebook}")
print(f"Then use: {follow_up_notebook}")
print(reasoning)

## Wrap-Up

By the end of this notebook, you should be able to:

- inspect the raw experiment data
- reproduce the cleaning logic
- compare conversion performance
- interpret a frequentist test carefully
- plan a future experiment using sample size and runtime estimates
- choose the right workflow for a real product question
